This notebook compares the performance of our IL-2 model among different conditions (dataset, IL-2 variant, cell type)

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.optimize import curve_fit
from src.simulate_func import hill_func
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

In [2]:
# Import datasets
df_sim_data = pd.read_csv('data/signaling/IL2_data.csv')
df_sim_mech = pd.read_csv("results/simulations_IL2s_fit.csv", index_col=0)
df_EC50_AMP = pd.read_csv('data/signaling/IL2_data_hill.csv')
cell_list_train = ["T8","T8.naive","Tregs","NK","T4_IL2RAlow","ACH-000061"]
cell_list_test = ["T8.M", "T8.naive", "T4.M_IL2RAlow", "T4.naive_IL2RAlow", "mTregs", "nTregs","ACH-000061","CTLL-2"]

# Divide data into train and tests datasets
df_sim_data_train = df_sim_data.loc[df_sim_data["Cell_type"].isin(cell_list_train)]
df_sim_data_train = df_sim_data_train.reset_index()[df_sim_data_train.columns]
df_EC50_AMP_train = df_EC50_AMP.loc[df_EC50_AMP["Cell_type"].isin(cell_list_train)]
df_EC50_AMP_train = df_EC50_AMP_train.reset_index()[df_EC50_AMP_train.columns]
df_sim_data_test = df_sim_data.loc[(df_sim_data["Cell_type"].isin(cell_list_test) & (df_sim_data["Variant"]=="WT"))]
df_sim_data_test = df_sim_data_test.reset_index()[df_sim_data_test.columns]
df_EC50_AMP_test = df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"].isin(cell_list_test) & (df_EC50_AMP["Variant"]=="WT"))]
df_EC50_AMP_test = df_EC50_AMP_test.reset_index()[df_EC50_AMP_test.columns]

In [3]:
# Error in training set
for plot_num in df_sim_data_train["Plot"].drop_duplicates():
    for variant in df_sim_data_train.loc[df_sim_data_train["Plot"]==plot_num,"Variant"].drop_duplicates():
        if variant == "WT":
            pSTAT5 = df_sim_mech.loc[(df_sim_mech["Plot"]==plot_num) & (df_sim_mech["Variant"]==variant),"Result"].values
            IL2 = df_sim_mech.loc[(df_sim_mech["Plot"]==plot_num) & (df_sim_mech["Variant"]==variant),"IL0"].values
            max_WT = np.max(pSTAT5)
            pSTAT5 = pSTAT5/max_WT*100
            fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([0,min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
            df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
            df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0]
        else:
            pSTAT5 = df_sim_mech.loc[(df_sim_mech["Plot"]==plot_num) & (df_sim_mech["Variant"]==variant),"Result"].values
            IL2 = df_sim_mech.loc[(df_sim_mech["Plot"]==plot_num) & (df_sim_mech["Variant"]==variant),"IL0"].values
            pSTAT5 = pSTAT5/max_WT*100
            fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([0,min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
            df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
            df_EC50_AMP_train.loc[(df_EC50_AMP_train["Plot"]==plot_num)&(df_EC50_AMP_train["Variant"]==variant),"Amp_m"] = fit[0] 
# Save errors
df_EC50_AMP_train["EC50_e"] = (df_EC50_AMP_train["EC50_t"]-df_EC50_AMP_train["EC50_m"]).abs()
df_EC50_AMP_train["Amp_e"] = (df_EC50_AMP_train["Amp_t"]-df_EC50_AMP_train["Amp_m"]).abs()

In [4]:
# Error in test set
for plot_num in df_sim_data_test["Plot"].drop_duplicates():
    for variant in df_sim_data_test.loc[df_sim_data_test["Plot"]==plot_num,"Variant"].drop_duplicates():
        if variant == "WT":
            pSTAT5 = df_sim_mech.loc[(df_sim_mech["Plot"]==plot_num) & (df_sim_mech["Variant"]==variant),"Result"].values
            IL2 = df_sim_mech.loc[(df_sim_mech["Plot"]==plot_num) & (df_sim_mech["Variant"]==variant),"IL0"].values
            max_WT = np.max(pSTAT5)
            pSTAT5 = pSTAT5/max_WT*100
            fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([0,min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
            df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
            df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"Amp_m"] = fit[0]
        else:
            pSTAT5 = df_sim_mech.loc[(df_sim_mech["Plot"]==plot_num) & (df_sim_mech["Variant"]==variant),"Result"].values
            IL2 = df_sim_mech.loc[(df_sim_mech["Plot"]==plot_num) & (df_sim_mech["Variant"]==variant),"IL0"].values
            pSTAT5 = pSTAT5/max_WT*100
            fit, cov = curve_fit(hill_func, IL2, pSTAT5, bounds = ([0,min(IL2)/10], [max(pSTAT5)+10, max(IL2)*10]))
            df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"EC50_m"] = np.log10(fit[1])
            df_EC50_AMP_test.loc[(df_EC50_AMP_test["Plot"]==plot_num)&(df_EC50_AMP_test["Variant"]==variant),"Amp_m"] = fit[0] 
# Save errors
df_EC50_AMP_test["EC50_e"] = (df_EC50_AMP_test["EC50_t"]-df_EC50_AMP_test["EC50_m"]).abs()
df_EC50_AMP_test["Amp_e"] = (df_EC50_AMP_test["Amp_t"]-df_EC50_AMP_test["Amp_m"]).abs()

### Is there a significant difference in the error between train and test sets?

In [156]:
f_statistic, p_value = stats.ttest_ind(df_EC50_AMP_train["EC50_e"].dropna().values, df_EC50_AMP_test["EC50_e"].dropna().values, equal_var=False, alternative="less")
fig, ax = plt.subplots(1, 1, figsize=(4.5, 7), dpi=400)

ax.boxplot(
    [
        df_EC50_AMP_train["EC50_e"].dropna().to_list(),
        df_EC50_AMP_test["EC50_e"].dropna().to_list()
    ],
    widths=0.6,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)
for i, values in enumerate([
        df_EC50_AMP_train["EC50_e"].dropna().to_list(),
        df_EC50_AMP_test["EC50_e"].dropna().to_list()
    ], start=1):
    x = np.random.normal(i, 0.06, size=len(values))
    ax.scatter(
        x,
        values,
        facecolor="white",
        edgecolor="black",
        linewidth=1.2,
        alpha=1,
        s=55,
        zorder=3
    )

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{EC50}$", fontsize=20)
ax.set_xticklabels(["Training set", "Test set"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "figures/stat_tests/Plot_box_EC50_train_test.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("Train vs test EC50 p value: "+str(p_value))
print("N: "+str([len(df_EC50_AMP_train["EC50_e"].dropna().values),len(df_EC50_AMP_test["EC50_e"].dropna().values)]))

Train vs test EC50 p value: 0.5667083445486338
N: [27, 8]


In [157]:
# Is there a significant difference in the error between train and test sets
f_statistic, p_value = stats.ttest_ind(df_EC50_AMP_train["Amp_e"].dropna().values, df_EC50_AMP_test["Amp_e"].dropna().values, equal_var=False, alternative="less")
fig, ax = plt.subplots(1, 1, figsize=(4.5, 7), dpi=400)

ax.boxplot(
    [
        df_EC50_AMP_train["Amp_e"].dropna().to_list(),
        df_EC50_AMP_test["Amp_e"].dropna().to_list()
    ],
    widths=0.6,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)
for i, values in enumerate([
        df_EC50_AMP_train["Amp_e"].dropna().to_list(),
        df_EC50_AMP_test["Amp_e"].dropna().to_list()
    ], start=1):
    x = np.random.normal(i, 0.06, size=len(values))
    ax.scatter(
        x,
        values,
        facecolor="white",
        edgecolor="black",
        linewidth=1.2,
        alpha=1,
        s=55,
        zorder=3
    )

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{AMP}$", fontsize=20)
ax.set_xticklabels(["Training set", "Test set"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "figures/stat_tests/Plot_box_AMP_train_test.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("Train vs test AMP p value: "+str(p_value))
print("N: "+str([len(df_EC50_AMP_train["Amp_e"].dropna().values),len(df_EC50_AMP_test["Amp_e"].dropna().values)]))

Train vs test AMP p value: 0.8871962452574521
N: [27, 8]


### Is there a significant difference in the error between cell types?

In [158]:
df_EC50_AMP = pd.concat([df_EC50_AMP_train,df_EC50_AMP_test])
df_EC50_AMP.loc[df_EC50_AMP["Cell_type"].isin(["T8","T8.M","T8.naive"]),"Cell_type"] = "T8"
df_EC50_AMP.loc[df_EC50_AMP["Cell_type"].isin(["Tregs","mTregs","nTregs"]),"Cell_type"] = "Tregs"

In [160]:
cell_types_EC50 = [df_EC50_AMP.loc[df_EC50_AMP["Cell_type"]==cell,"EC50_e"].dropna().to_list() for cell in ['T8','Tregs','NK']]
f_statistic, p_value = stats.f_oneway(cell_types_EC50[0], cell_types_EC50[1], cell_types_EC50[2])
fig, ax = plt.subplots(1, 1, figsize=(6, 7), dpi=400)

ax.boxplot(
    cell_types_EC50,
    widths=0.6,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)
for i, values in enumerate(cell_types_EC50, start=1):
    x = np.random.normal(i, 0.06, size=len(values))
    ax.scatter(
        x,
        values,
        facecolor="white",
        edgecolor="black",
        linewidth=1.2,
        alpha=1,
        s=55,
        zorder=3
    )

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{EC50}$", fontsize=20)
ax.set_xticklabels(["CD8⁺\nT cells", "Regulatory\nT cells","NK cells"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "figures/stat_tests/Plot_box_EC50_cells.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)

print("Cell type EC50 p value: "+str(p_value))
print("N: "+str([len(cell_types_EC50[0]), len(cell_types_EC50[1]), len(cell_types_EC50[2])]))

Cell type EC50 p value: 0.9049746263407477
N: [12, 10, 7]


In [163]:
cell_types_AMP = [df_EC50_AMP.loc[df_EC50_AMP["Cell_type"]==cell,"Amp_e"].dropna().to_list() for cell in ['T8','Tregs','NK']]
f_statistic, p_value = stats.f_oneway(cell_types_AMP[0], cell_types_AMP[1], cell_types_AMP[2])
fig, ax = plt.subplots(1, 1, figsize=(6, 7), dpi=400)

ax.boxplot(
    cell_types_AMP,
    widths=0.6,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)
for i, values in enumerate(cell_types_AMP, start=1):
    x = np.random.normal(i, 0.06, size=len(values))
    ax.scatter(
        x,
        values,
        facecolor="white",
        edgecolor="black",
        linewidth=1.2,
        alpha=1,
        s=55,
        zorder=3
    )

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{AMP}$", fontsize=20)
ax.set_xticklabels(["CD8⁺\nT cells", "Regulatory\nT cells", "NK cells"], fontsize=17)

plt.tight_layout()
plt.savefig(
    "figures/stat_tests/Plot_box_AMP_cells.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)

print("Cell type EC50 p value: "+str(p_value))
print("N: "+str([len(cell_types_AMP[0]), len(cell_types_AMP[1]), len(cell_types_AMP[2])]))

Cell type EC50 p value: 0.152771272397717
N: [12, 10, 7]


### Is there a significant difference in the error between IL-10 variants?

In [74]:
variant_EC50 = [df_EC50_AMP.loc[df_EC50_AMP["Variant"]==var,"EC50_e"].dropna().to_list() for var in df_EC50_AMP["Variant"].drop_duplicates()]
f_statistic, p_value = stats.f_oneway(variant_EC50[0], variant_EC50[1], variant_EC50[2])
fig, ax = plt.subplots(1, 1, figsize=(8, 7), dpi=400)

ax.boxplot(
    variant_EC50,
    widths=0.6,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)
for i, values in enumerate(variant_EC50, start=1):
    x = np.random.normal(i, 0.06, size=len(values))
    ax.scatter(
        x,
        values,
        facecolor="white",
        edgecolor="black",
        linewidth=1.2,
        alpha=1,
        s=55,
        zorder=3
    )

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{EC50}$", fontsize=20)
ax.set_xticklabels(['WT IL-2','NKTR-358','MDNA11','ALKS4230','THOR-707'], fontsize=17)

plt.tight_layout()
plt.savefig(
    "figures/stat_tests/Plot_box_EC50_variants.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)
print("Variant EC50 p value: "+str(p_value))
print("N: "+str([len(variant_EC50[0]), len(variant_EC50[1]), len(variant_EC50[2]), len(variant_EC50[3]), len(variant_EC50[4])]))

Variant EC50 p value: 0.667775239024165
N: [23, 2, 3, 4, 3]


In [75]:
variant_AMP = [df_EC50_AMP.loc[df_EC50_AMP["Variant"]==var,"Amp_e"].dropna().to_list() for var in ['NKTR-358','MDNA11','ALKS4230','THOR-707']]
f_statistic, p_value = stats.f_oneway(variant_AMP[0], variant_AMP[1])
fig, ax = plt.subplots(1, 1, figsize=(8, 7), dpi=400)

ax.boxplot(
    variant_AMP,
    widths=0.6,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor="grey", edgecolor="black", linewidth=1.5),
    medianprops=dict(color="black", linewidth=1.5),
    whiskerprops=dict(color="black", linewidth=1.5),
    capprops=dict(color="black", linewidth=1.5)
)
for i, values in enumerate(variant_AMP, start=1):
    x = np.random.normal(i, 0.06, size=len(values))
    ax.scatter(
        x,
        values,
        facecolor="white",
        edgecolor="black",
        linewidth=1.2,
        alpha=1,
        s=55,
        zorder=3
    )

# Spine formatting
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Tick formatting
ax.xaxis.set_tick_params(width=4, length=8)
ax.yaxis.set_tick_params(width=4, length=8)
ax.tick_params(axis="x", labelsize=17)
ax.tick_params(axis="y", labelsize=17)

# Labels
ax.set_ylabel("$MAE_{AMP}$", fontsize=20)
ax.set_xticklabels(['NKTR-358','MDNA11','ALKS4230','THOR-707'], fontsize=17)

plt.tight_layout()
plt.savefig(
    "figures/stat_tests/Plot_box_AMP_variants.pdf",
    transparent=True,
    bbox_inches="tight"
)
plt.close(fig)

print("Variant AMP p value: "+str(p_value))
print("N: "+str([len(variant_AMP[0]), len(variant_AMP[1]), len(variant_AMP[2]), len(variant_AMP[3])]))

Variant AMP p value: 0.6762255687863733
N: [2, 3, 4, 3]


In [107]:
groups_keep = ['Tregs__WT','NK__WT']
df_EC50_AMP_train.assign(group=df_EC50_AMP_train["Cell_type"] + "__" + df_EC50_AMP_train["Variant"]).loc[lambda d: d["group"].isin(groups_keep)].groupby("group").var(numeric_only=True)["EC50_t"].mean()

0.2980113800991831

### What is the variability of the train+test dataset on the same condition?

In [175]:
# Cluster cell subtypes and get the variability of the EC50
df_EC50_AMP = pd.read_csv('data/signaling/IL2_data_hill.csv')
df_EC50_AMP = df_EC50_AMP.loc[df_EC50_AMP["Variant"]=="WT"]
df_EC50_AMP.loc[df_EC50_AMP["Cell_type"].isin(["T8","T8.M","T8.naive"]),"Cell_type"] = "T8"
df_EC50_AMP.loc[df_EC50_AMP["Cell_type"].isin(["T4.M_IL2RAlow","T4.naive_IL2RAlow","T4_IL2RAlow"]),"Cell_type"] = "T4_IL2RAlow"

df_EC50_AMP.loc[df_EC50_AMP["Cell_type"].isin(["Tregs","mTregs","nTregs"]),"Cell_type"] = "Tregs"
print("Mean EC50 std. dev. per condition: "+str(df_EC50_AMP.loc[df_EC50_AMP["Cell_type"].isin(['T8','Tregs','NK','T4_IL2RAlow'])].groupby("Cell_type")["EC50_t"].std().mean()))

Mean EC50 std. dev. per condition: 0.43481923180945603
